# LendingClub PD EDA

Structured exploratory analysis using method-chaining style transformations.

## Data Loading

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from src.plot_style import apply_plot_style

apply_plot_style()
sns.set_theme(style="whitegrid")

data_path = "data/lending_club_sample.csv"
df = (
    pd.read_csv(data_path, low_memory=False)
    .assign(
        issue_d=lambda d: pd.to_datetime(d["issue_d"], errors="coerce"),
        loan_status=lambda d: d["loan_status"].astype(str),
        default_flag=lambda d: d["loan_status"].eq("Charged Off").astype(int),
        grade=lambda d: pd.Categorical(d["grade"], categories=list("ABCDEFG"), ordered=True)
        if "grade" in d.columns else d.get("grade")
    )
)
df.head()

## Missingness Audit

In [ ]:
missingness = (
    df
    .isna()
    .mean()
    .rename("missing_fraction")
    .reset_index(names="feature")
    .sort_values("missing_fraction", ascending=False)
)
missingness.head(20)

In [ ]:
(
    missingness
    .query("missing_fraction > 0")
    .head(20)
    .set_index("feature")
    .plot(kind="bar", legend=False, title="Top Feature Missingness")
);
plt.ylabel("Missing Fraction")
plt.tight_layout()

## Univariate Distributions

In [ ]:
numeric_cols = (
    df
    .select_dtypes(include=["number"])
    .columns
    .difference(["default_flag"])
    .tolist()
)
(
    df[numeric_cols[:6]]
    .hist(bins=30, figsize=(14, 10))
)
plt.suptitle("Numeric Feature Distributions (sample)")
plt.tight_layout()

In [ ]:
if "grade" in df.columns:
    (
        df["grade"]
        .value_counts(dropna=False)
        .sort_index()
        .plot(kind="bar", title="Grade Distribution")
    )
    plt.tight_layout()

## Bivariate Analysis: Default Rate by Grade, Purpose, Home Ownership

In [ ]:
for col in ["grade", "purpose", "home_ownership"]:
    if col in df.columns:
        summary = (
            df
            .groupby(col, dropna=False)
            .agg(default_rate=("default_flag", "mean"), n=("default_flag", "size"))
            .sort_values("default_rate", ascending=False)
        )
        display(summary.head(20))
        summary["default_rate"].plot(kind="bar", title=f"Default Rate by {col}")
        plt.ylabel("Default Rate")
        plt.tight_layout()
        plt.show()

## Pivot: Charge-off Rate by Grade × Purpose

In [ ]:
if {"grade", "purpose"}.issubset(df.columns):
    pivot = (
        df
        .pivot_table(
            index="grade",
            columns="purpose",
            values="default_flag",
            aggfunc="mean",
            margins=True,
            margins_name="Overall"
        )
        .sort_index()
    )
    display(pivot)

## Monthly Default Rate Time Series (3-month Rolling Average)

In [ ]:
monthly_defaults = (
    df
    .dropna(subset=["issue_d"])
    .set_index("issue_d")
    .resample("M")
    .agg(default_rate=("default_flag", "mean"))
    .assign(rolling_3m=lambda d: d["default_rate"].rolling(3).mean())
)
ax = monthly_defaults.plot(title="Monthly Default Rate with 3-Month Rolling Average")
ax.set_ylabel("Default Rate")
plt.tight_layout()

## Correlation Heatmap (Masked Upper Triangle)

In [ ]:
corr = df.select_dtypes(include=["number"]).corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(14, 10))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, linewidths=0.2)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()